In [4]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os
from glob import glob
os.chdir(r"C:\Users\Tomasz Ostrzyżek\Desktop\STUDIA\Magisterka\Semestr_1\ZAW\PROJEKT")

In [5]:
K_wady = [
    "bent_wire",
    "cable_swap",
    "combined",
    "cut_inner_insulation",
    "cut_outer_insulation",
    "good",
    "missing_cable",
    "missing_wire",
    "poke_insulation",
]

K_aktywna = "cable_swap"

test_file  = f"cable-defect-detection/data/test/{K_aktywna}/*.png"
gt_file    = f"cable-defect-detection/data/ground_truth/{K_aktywna}/*_mask.png"
good_file = "cable-defect-detection/data/test/good/*.png"

test_files = glob(test_file)
good_files = glob(good_file)
gt_files = glob(gt_file)

print("Liczba obrazów testowych:", len(test_files))
print("Liczba obrazów good:", len(good_files))
print("Liczba masek ground truth:", len(gt_files))


Liczba obrazów testowych: 11
Liczba obrazów good: 46
Liczba masek ground truth: 11


In [ ]:
kernel = np.ones((7,7), np.uint8)
scale = 0.25

TH_H = 2*20      # próg na różnicę odcienia
TH_S = 2*30      # minimalne nasycenie, żeby H miało sens
MIN_AREA = 150
MAX_AREA = 50000

def wybierz_najlepszy_good(I_bad, good_files, scale):
    height, width = I_bad.shape[:2]
    I_bad_small = cv2.resize(I_bad, (int(scale * width), int(scale * height)))
    I_bad_blur = cv2.medianBlur(I_bad_small, 5)
    bad_gray = cv2.cvtColor(I_bad_blur, cv2.COLOR_BGR2GRAY)

    h, w = bad_gray.shape[:2]
    x1, x2 = int(0.2 * w), int(0.8 * w)
    y1, y2 = int(0.2 * h), int(0.8 * h)

    bad_roi = bad_gray[y1:y2, x1:x2]

    best_score = 1e18
    best_good_color = None
    best_good_blur = None

    for gf in good_files:
        I_good = cv2.imread(gf)
        if I_good is None:
            continue

        hg, wg = I_good.shape[:2]
        I_good_small = cv2.resize(I_good, (int(scale * wg), int(scale * hg)))
        I_good_blur = cv2.medianBlur(I_good_small, 5)
        good_gray = cv2.cvtColor(I_good_blur, cv2.COLOR_BGR2GRAY)

        good_roi = good_gray[y1:y2, x1:x2]

        D = cv2.absdiff(bad_roi, good_roi)
        score = np.mean(D)

        if score < best_score:
            best_score = score
            best_good_color = I_good_small
            best_good_blur = I_good_blur

    return best_good_color, best_good_blur, best_score

def hue_diff(h1, h2):
    d = cv2.absdiff(h1, h2)
    return np.minimum(d, 180 - d)

for f in test_files:
    I = cv2.imread(f)

    if I is None:
        continue
    
    I_good_color, I_good_blur, best_score = wybierz_najlepszy_good(I, good_files, scale)
    HSV_good = cv2.cvtColor(I_good_blur, cv2.COLOR_BGR2HSV)
    H_good, S_good, V_good = cv2.split(HSV_good)
    
    height, width = I.shape[:2]
    I_wada = cv2.resize(I, (int(scale * width), int(scale * height)))
    I_vis = I_wada.copy()

    I_blur = cv2.medianBlur(I_wada, 5)
    HSV_bad = cv2.cvtColor(I_blur, cv2.COLOR_BGR2HSV)
    H_bad, S_bad, V_bad = cv2.split(HSV_bad)

    # ROI (środek obrazu)
    H_bad_roi  = H_bad
    H_good_roi = H_good
    S_bad_roi  = S_bad
    S_good_roi = S_good
    x1, y1 = 0, 0

    # różnica H liczona kołowo
    D = hue_diff(H_bad_roi, H_good_roi)
    D_sat = cv2.absdiff(S_bad_roi, S_good_roi)
    # tylko tam, gdzie kolor jest sensowny
    valid_mask = ((S_bad_roi > TH_S) | (S_good_roi > TH_S)).astype(np.uint8) * 255

    # progowanie różnicy H
    _, thresh = cv2.threshold(D, TH_H, 255, cv2.THRESH_BINARY)
    _, thresh_s = cv2.threshold(D_sat, TH_S, 255, cv2.THRESH_BINARY)
    # usunięcie pikseli o małym nasyceniu
    thresh = cv2.bitwise_or(thresh, thresh_s)
    thresh = cv2.bitwise_and(thresh, valid_mask)

    # morfologia
    otwarcie = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)
    domkniecie = cv2.morphologyEx(otwarcie, cv2.MORPH_CLOSE, kernel)

    # pełne obrazy do wizualizacji
    D_full       = D
    thresh_full  = thresh
    mask_full    = domkniecie
    satmask_full = valid_mask

    retval, labels, stats, centroids = cv2.connectedComponentsWithStats(domkniecie)

    if retval > 1:
        tab = stats[1:, cv2.CC_STAT_AREA]
        pi = np.argmax(tab) + 1

        x = stats[pi, cv2.CC_STAT_LEFT]
        y = stats[pi, cv2.CC_STAT_TOP]
        ww = stats[pi, cv2.CC_STAT_WIDTH]
        hh = stats[pi, cv2.CC_STAT_HEIGHT]
        area = stats[pi, cv2.CC_STAT_AREA]

        if MIN_AREA < area:
            cv2.rectangle(I_vis, (x + x1, y + y1), (x + ww + x1, y + hh + y1), (0, 255, 255), 2)
            cv2.putText(I_vis, f"{area}", (x + x1, y + y1 - 10),cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
                      
    cv2.imshow("Domkniecie", domkniecie)
    cv2.imshow("Wada", I_wada)
    cv2.imshow("Wzorzec - GOOD", I_good_color)
    cv2.imshow("Hue difference", D_full)
    cv2.imshow("Valid saturation mask", satmask_full)
    cv2.imshow("Thresh H", thresh_full)
    cv2.imshow("Mask H", mask_full)
    cv2.imshow("Wykrycie HSV", I_vis)

    key = cv2.waitKey(0) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('d'):
        cv2.destroyAllWindows()  # zamknij bieżące, pętla idzie dalej

cv2.destroyAllWindows()